[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/18_montecarlo_search/notebooks/03_uct_y_experimentos.ipynb)

# Notebook 3: UCT y experimentos

**Módulo 18 — Monte Carlo Tree Search · Notebook 03**

---

## ¿De qué trata este notebook?

En este notebook implementamos **MCTS con UCT** y lo comparamos experimentalmente
con la versión naive (selección greedy). Exploramos el efecto de la constante
de exploración $c$ y del presupuesto de iteraciones.

**Lo que vas a construir y explorar:**

1. La fórmula UCT y su implementación.
2. Comparación UCT vs naive en Hex 3×3.
3. Experimento: presupuesto de iteraciones vs tasa de victorias.
4. Experimento: efecto de la constante $c$.
5. Gráficas de resultados.

**Relación con las notas de clase:** sección 04 (UCT: la conexión con bandidos).

**Prerequisitos:** Notebooks 01-02, módulo 17 (bandidos multibrazo).

---

In [ ]:
# Solo en Colab
# !pip install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import random
import copy
from collections import deque

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "blue": "#2E86AB",
    "red": "#E94F37",
    "green": "#27AE60",
    "gray": "#7F8C8D",
    "orange": "#F39C12",
    "purple": "#8E44AD",
}

random.seed(42)
np.random.seed(42)

---

## 1. Clase `Hex` (autocontenido)

In [ ]:
class Hex:
    """Juego de Hex en un tablero de size x size."""

    NEIGHBORS = [(-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0)]

    def __init__(self, size=7):
        self.size = size
        self.board = [[0] * size for _ in range(size)]
        self.current_player = 1

    def actions(self):
        return [(r, c) for r in range(self.size)
                for c in range(self.size) if self.board[r][c] == 0]

    def result(self, action):
        new = copy.deepcopy(self)
        r, c = action
        new.board[r][c] = self.current_player
        new.current_player = 3 - self.current_player
        return new

    def is_terminal(self):
        return self._has_path(1) or self._has_path(2)

    def utility(self, player):
        if self._has_path(player):
            return 1
        if self._has_path(3 - player):
            return -1
        return 0

    def _has_path(self, player):
        n = self.size
        if player == 1:
            start = [(0, c) for c in range(n) if self.board[0][c] == 1]
            goal = lambda r, c: r == n - 1
        else:
            start = [(r, 0) for r in range(n) if self.board[r][0] == 2]
            goal = lambda r, c: c == n - 1
        visited = set()
        queue = deque(start)
        while queue:
            r, c = queue.popleft()
            if (r, c) in visited:
                continue
            visited.add((r, c))
            if goal(r, c):
                return True
            for dr, dc in self.NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and self.board[nr][nc] == player:
                    queue.append((nr, nc))
        return False

---

## 2. MCTS con UCT

La fórmula UCT reemplaza la selección greedy ($Q/N$) con:

$$\text{UCT}(v) = \frac{Q(v)}{N(v)} + c \sqrt{\frac{\ln N(\text{padre})}{N(v)}}$$

El primer término es **explotación** (favorece nodos con buenos resultados).
El segundo es **exploración** (favorece nodos poco visitados). La constante
$c$ controla el balance.

Es exactamente UCB1 del módulo 17 aplicado a cada nodo del árbol.

In [ ]:
class MCTSNode:
    """Nodo del árbol de MCTS."""

    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = {}
        self.N = 0
        self.Q = 0.0
        self.unexpanded = list(state.actions()) if not state.is_terminal() else []

    def win_rate(self):
        return self.Q / self.N if self.N > 0 else 0.0


def uct_value(child, parent_visits, c):
    """Calcula el valor UCT de un nodo hijo.

    UCT(v) = Q(v)/N(v) + c * sqrt(ln(N_padre) / N(v))

    Si el hijo no ha sido visitado, retorna infinito (prioridad máxima).
    """
    if child.N == 0:
        return float('inf')
    exploitation = child.Q / child.N
    exploration = c * math.sqrt(math.log(parent_visits) / child.N)
    return exploitation + exploration


def mcts_uct(game_state, iterations, player, c=1.41):
    """MCTS con selección UCT.

    Parameters
    ----------
    game_state : Hex
        Estado actual del juego.
    iterations : int
        Presupuesto de iteraciones.
    player : int
        Jugador desde cuya perspectiva medimos utilidad.
    c : float
        Constante de exploración (default: sqrt(2) ~ 1.41).

    Returns
    -------
    best_action, root
    """
    root = MCTSNode(game_state)

    for _ in range(iterations):
        node = root

        # [M1] Selección con UCT                              [U1]
        while not node.unexpanded and node.children:
            node = max(node.children.values(),
                       key=lambda ch: uct_value(ch, node.N, c))

        # [M2] Expansión
        if node.unexpanded:
            action = node.unexpanded.pop()
            child_state = node.state.result(action)
            child = MCTSNode(child_state, parent=node)
            node.children[action] = child
            node = child

        # [M3] Simulación
        sim = copy.deepcopy(node.state)
        while not sim.is_terminal():
            sim = sim.result(random.choice(sim.actions()))
        reward = sim.utility(player)

        # [M4] Retropropagación
        while node is not None:
            node.N += 1
            node.Q += reward
            node = node.parent

    best_action = max(root.children, key=lambda a: root.children[a].N)
    return best_action, root


def mcts_naive(game_state, iterations, player):
    """MCTS vanilla con selección greedy (Q/N)."""
    root = MCTSNode(game_state)

    for _ in range(iterations):
        node = root

        while not node.unexpanded and node.children:
            node = max(node.children.values(), key=lambda ch: ch.Q / ch.N)

        if node.unexpanded:
            action = node.unexpanded.pop()
            child_state = node.state.result(action)
            child = MCTSNode(child_state, parent=node)
            node.children[action] = child
            node = child

        sim = copy.deepcopy(node.state)
        while not sim.is_terminal():
            sim = sim.result(random.choice(sim.actions()))
        reward = sim.utility(player)

        while node is not None:
            node.N += 1
            node.Q += reward
            node = node.parent

    best_action = max(root.children, key=lambda a: root.children[a].N)
    return best_action, root

---

## 3. Comparación: UCT vs naive en Hex 3×3

Comparemos la distribución de visitas en los hijos de la raíz con las
dos políticas de selección.

In [ ]:
state = Hex(size=3)
M = 500

# Ejecutar ambas versiones con la misma semilla
random.seed(42)
best_naive, root_naive = mcts_naive(state, iterations=M, player=1)

random.seed(42)
best_uct, root_uct = mcts_uct(state, iterations=M, player=1, c=1.41)

# Comparar distribuciones de visitas
acciones = sorted(root_uct.children.keys())
etiquetas = [f"({r},{c})" for r, c in acciones]

visitas_naive = [root_naive.children.get(a, MCTSNode(state)).N for a in acciones]
visitas_uct = [root_uct.children.get(a, MCTSNode(state)).N for a in acciones]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

ax1.bar(etiquetas, visitas_naive, color=COLORS["red"], alpha=0.8)
ax1.set_title(f"Naive (greedy Q/N)\nEligió: {best_naive}", fontsize=12)
ax1.set_ylabel("Visitas (N)")
ax1.set_xlabel("Celda")
ax1.tick_params(axis='x', rotation=45)

ax2.bar(etiquetas, visitas_uct, color=COLORS["blue"], alpha=0.8)
ax2.set_title(f"UCT (c = 1.41)\nEligió: {best_uct}", fontsize=12)
ax2.set_xlabel("Celda")
ax2.tick_params(axis='x', rotation=45)

plt.suptitle(f"Distribución de visitas — {M} iteraciones en Hex 3×3",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Naive eligió: {best_naive} | UCT eligió: {best_uct}")
print(f"\nVisitas naive: min={min(visitas_naive)}, max={max(visitas_naive)}")
print(f"Visitas UCT:   min={min(visitas_uct)}, max={max(visitas_uct)}")
print(f"\nUCT distribuye las visitas de forma más equilibrada.")

---

## 4. Enfrentamiento directo: UCT vs naive

Hagamos jugar ambos agentes entre sí en partidas completas de Hex 3×3.
Cada agente usa MCTS para decidir su movimiento en cada turno.

In [ ]:
def agente_mcts_uct(state, player, iterations=200, c=1.41):
    """Agente que usa MCTS con UCT para elegir movimiento."""
    action, _ = mcts_uct(state, iterations, player, c)
    return action


def agente_mcts_naive(state, player, iterations=200):
    """Agente que usa MCTS naive para elegir movimiento."""
    action, _ = mcts_naive(state, iterations, player)
    return action


def agente_aleatorio(state, player):
    """Agente que elige al azar."""
    return random.choice(state.actions())


def jugar_partida(size, agente1, agente2):
    """Juega una partida entre dos agentes.

    agente1 juega como Negro (1), agente2 como Blanco (2).
    Retorna el ganador: 1 o 2.
    """
    state = Hex(size=size)
    while not state.is_terminal():
        if state.current_player == 1:
            action = agente1(state, player=1)
        else:
            action = agente2(state, player=2)
        state = state.result(action)
    return 1 if state.utility(1) == 1 else 2


# UCT vs Naive: 50 partidas (cada uno juega 25 como Negro)
random.seed(42)

n_partidas = 25
iters_por_movimiento = 200

# UCT como Negro vs Naive como Blanco
uct_negro_gana = 0
for i in range(n_partidas):
    ganador = jugar_partida(
        size=3,
        agente1=lambda s, p: agente_mcts_uct(s, p, iters_por_movimiento),
        agente2=lambda s, p: agente_mcts_naive(s, p, iters_por_movimiento)
    )
    if ganador == 1:
        uct_negro_gana += 1
    if (i + 1) % 5 == 0:
        print(f"  Partidas {i+1}/{n_partidas} completadas (UCT como Negro)...")

# Naive como Negro vs UCT como Blanco
naive_negro_gana = 0
for i in range(n_partidas):
    ganador = jugar_partida(
        size=3,
        agente1=lambda s, p: agente_mcts_naive(s, p, iters_por_movimiento),
        agente2=lambda s, p: agente_mcts_uct(s, p, iters_por_movimiento)
    )
    if ganador == 1:
        naive_negro_gana += 1
    if (i + 1) % 5 == 0:
        print(f"  Partidas {i+1}/{n_partidas} completadas (Naive como Negro)...")

uct_total = uct_negro_gana + (n_partidas - naive_negro_gana)
naive_total = (n_partidas - uct_negro_gana) + naive_negro_gana
total = 2 * n_partidas

print(f"\n{'='*50}")
print(f"Resultados ({total} partidas, {iters_por_movimiento} iter/movimiento):")
print(f"  UCT gana: {uct_total}/{total} ({100*uct_total/total:.1f}%)")
print(f"  Naive gana: {naive_total}/{total} ({100*naive_total/total:.1f}%)")

---

## 5. Experimento: presupuesto de iteraciones vs tasa de victorias

¿Cuántas iteraciones necesita MCTS con UCT para ganarle consistentemente
al agente aleatorio? Variamos el presupuesto y medimos la tasa de victorias.

In [ ]:
random.seed(42)

presupuestos = [10, 25, 50, 100, 200, 500]
n_partidas = 30  # partidas por presupuesto
tasas = []

for M in presupuestos:
    victorias = 0
    for _ in range(n_partidas):
        ganador = jugar_partida(
            size=3,
            agente1=lambda s, p, m=M: agente_mcts_uct(s, p, m),
            agente2=agente_aleatorio
        )
        if ganador == 1:
            victorias += 1
    tasa = victorias / n_partidas
    tasas.append(tasa)
    print(f"  M={M:4d}: UCT gana {victorias}/{n_partidas} = {tasa*100:.0f}%")

# Gráfica
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(presupuestos, tasas, color=COLORS["blue"], marker='o', linewidth=2, markersize=8)
ax.axhline(1.0, color=COLORS["green"], linestyle="--", alpha=0.5, label="100% victorias")
ax.axhline(0.5, color=COLORS["gray"], linestyle="--", alpha=0.5, label="50% (azar)")
ax.set_xlabel("Iteraciones por movimiento (M)")
ax.set_ylabel("Tasa de victorias vs aleatorio")
ax.set_title("MCTS (UCT) vs Aleatorio — Hex 3×3")
ax.set_xscale("log")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()

---

## 6. Experimento: efecto de la constante $c$

La constante de exploración $c$ controla el balance entre explotación y exploración.
Según la teoría (§17.3), el valor óptimo es $c = \sqrt{2} \approx 1.41$.
Verifiquémoslo empíricamente.

| Valor de $c$ | Efecto |
|:---:|---|
| $c \to 0$ | Pura explotación |
| $c = \sqrt{2}$ | Balance teórico |
| $c$ grande | Exceso de exploración |

In [ ]:
random.seed(42)

c_values = [0.01, 0.1, 0.5, 1.0, 1.41, 2.0, 3.0, 5.0]
n_partidas = 30
M = 100  # presupuesto fijo

tasas_c = []

for c in c_values:
    victorias = 0
    for _ in range(n_partidas):
        ganador = jugar_partida(
            size=3,
            agente1=lambda s, p, cv=c: agente_mcts_uct(s, p, M, cv),
            agente2=agente_aleatorio
        )
        if ganador == 1:
            victorias += 1
    tasa = victorias / n_partidas
    tasas_c.append(tasa)
    print(f"  c={c:5.2f}: UCT gana {victorias}/{n_partidas} = {tasa*100:.0f}%")

# Gráfica
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(c_values, tasas_c, color=COLORS["purple"], marker='s', linewidth=2, markersize=8)
ax.axvline(1.41, color=COLORS["orange"], linestyle="--", alpha=0.7, label="$c = \\sqrt{2} \\approx 1.41$")
ax.axhline(0.5, color=COLORS["gray"], linestyle="--", alpha=0.5)
ax.set_xlabel("Constante de exploración $c$")
ax.set_ylabel("Tasa de victorias vs aleatorio")
ax.set_title(f"Efecto de $c$ en MCTS (UCT) — Hex 3×3, M={M}")
ax.set_xscale("log")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()

mejor_c = c_values[np.argmax(tasas_c)]
print(f"\nMejor c encontrado: {mejor_c}")
print(f"Valor teórico óptimo: sqrt(2) = {math.sqrt(2):.3f}")

---

## 7. Visualización: distribución de visitas para distintos valores de $c$

Veamos cómo $c$ cambia la distribución de visitas en los hijos de la raíz.

In [ ]:
state = Hex(size=3)
c_demos = [0.01, 0.5, 1.41, 5.0]
M = 300

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for ax, c in zip(axes, c_demos):
    random.seed(42)
    best, root = mcts_uct(state, iterations=M, player=1, c=c)

    acciones = sorted(root.children.keys())
    visitas = [root.children[a].N for a in acciones]
    etiquetas = [f"({r},{c_})" for r, c_ in acciones]

    color = COLORS["red"] if c < 0.1 else (COLORS["blue"] if c < 3 else COLORS["orange"])
    ax.bar(etiquetas, visitas, color=color, alpha=0.8)
    ax.set_title(f"$c = {c}$\nEligió: {best}", fontsize=11)
    ax.tick_params(axis='x', rotation=90, labelsize=7)
    if ax == axes[0]:
        ax.set_ylabel("Visitas (N)")

plt.suptitle(f"Efecto de $c$ en la distribución de visitas — {M} iteraciones",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("c pequeño -> visitas concentradas (explotación pura)")
print("c grande  -> visitas dispersas (exploración excesiva)")
print("c ~1.41   -> balance adecuado")

---

## 8. Ejercicio: UCT en Hex 5×5

**Tu tarea:** repite los experimentos anteriores en Hex 5×5.

1. ¿Necesitas más iteraciones para que UCT funcione bien en un tablero más grande?
2. ¿El valor óptimo de $c$ cambia con el tamaño del tablero?
3. ¿Cuántas partidas necesitas para obtener estimaciones estables de la tasa de victorias?

**Nota:** Las partidas en 5×5 son más lentas. Reduce `n_partidas` si es necesario.

In [ ]:
# === TU CÓDIGO AQUÍ ===
# Sugerencia: empieza con n_partidas=10 y presupuestos=[50, 100, 200]

random.seed(42)

# 1. Presupuesto vs tasa de victorias en Hex 5×5
presupuestos_5x5 = [50, 100, 200]
n_partidas_5x5 = 10

print("Presupuesto vs victorias en Hex 5×5 (UCT vs aleatorio):\n")
for M in presupuestos_5x5:
    victorias = 0
    for _ in range(n_partidas_5x5):
        ganador = jugar_partida(
            size=5,
            agente1=lambda s, p, m=M: agente_mcts_uct(s, p, m),
            agente2=agente_aleatorio
        )
        if ganador == 1:
            victorias += 1
    print(f"  M={M:4d}: UCT gana {victorias}/{n_partidas_5x5}")

print("\n¿Qué observas? Escribe tu análisis aquí:")
# TU RESPUESTA:

---

## 9. Resumen

En este notebook vimos que:

1. **UCT = UCB1 en cada nodo**: la misma fórmula del módulo 17, aplicada a árboles.
2. **UCT distribuye mejor las visitas**: explora alternativas que la selección
   greedy ignora.
3. **Más iteraciones = mejor rendimiento**, con rendimientos decrecientes.
4. **La constante $c$** controla el balance exploración/explotación.
   $c = \sqrt{2}$ es un buen punto de partida.

En el **notebook 04** (torneo) enfrentaremos MCTS (UCT) contra otros agentes
en un torneo completo.

---